In [0]:
from datetime import datetime
import re

landing_path = "s3a://retail-etl-lakehouse/landing/"
archive_path = "s3a://retail-etl-lakehouse/archive/"

files = dbutils.fs.ls(landing_path)

file_groups = {}

for file in files:

    filename = file.name

    print("Checking:", filename)

    # Handles:
    # customers-src_19042026_100105.csv

    match = re.match(r'([a-z\-]+)_(\d{8})_(\d{6})\.csv', filename)

    if match:

        file_type = match.group(1)

        timestamp = datetime.strptime(
            match.group(2) + match.group(3),
            "%d%m%Y%H%M%S"
        )

        if file_type not in file_groups:
            file_groups[file_type] = []

        file_groups[file_type].append((file, timestamp))

for file_type in file_groups:

    sorted_files = sorted(
        file_groups[file_type],
        key=lambda x: x[1],
        reverse=True
    )

    latest_file = sorted_files[0][0]

    print("Latest File Kept:", latest_file.name)

    for old_file, ts in sorted_files[1:]:

        dbutils.fs.mv(
            old_file.path,
            archive_path + old_file.name
        )

        print("Archived:", old_file.name)